<a href="https://colab.research.google.com/github/rikardolord/hellblade-senuas-sacrifice-adult-enhancements/blob/branch/TARJAR_CPF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pytesseract PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 103.0 MB/s eta 0:00:00


In [3]:
!sudo apt-get install tesseract-ocr


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [5]:
import fitz  # PyMuPDF
import re
import pytesseract
from PIL import Image
import os

# Configuração do caminho do Tesseract OCR (se estiver rodando localmente)
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def tarjar_cpfs(caminho_pdf_original, caminho_pdf_final):
    """
    Processa um PDF não pesquisável, realiza OCR, encontra os CPFs
    e os oculta com tarjas pretas.
    """
    try:
        doc = fitz.open(caminho_pdf_original)
        novo_doc = fitz.open()

        regex_cpf = r'\d{3}\.?\d{3}\.?\d{3}-?\d{2}'

        for num_pagina in range(len(doc)):
            pagina = doc.load_page(num_pagina)
            pixmap = pagina.get_pixmap(dpi=300)
            img = Image.frombytes("RGB", [pixmap.width, pixmap.height], pixmap.samples)

            data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)

            nova_pagina = novo_doc.new_page(width=pagina.rect.width, height=pagina.rect.height)
            nova_pagina.insert_image(pagina.rect, pixmap=pixmap)

            for i in range(len(data['text'])):
                text = data['text'][i]
                if re.match(regex_cpf, text):
                    x, y, w, h = data['left'][i], data['top'][i], data['width'][i], data['height'][i]

                    escala_x = pagina.rect.width / img.width
                    escala_y = pagina.rect.height / img.height

                    rect = fitz.Rect(x * escala_x, y * escala_y, (x + w) * escala_x, (y + h) * escala_y)

                    nova_pagina.draw_rect(rect, color=(0, 0, 0), fill=(0, 0, 0), overlay=True)
                    print(f"CPF encontrado e tarjado na página {num_pagina + 1} de {caminho_pdf_original}: {text}")

        novo_doc.save(caminho_pdf_final)
        novo_doc.close()
        print(f"Novo arquivo salvo: {caminho_pdf_final}")
    except Exception as e:
        print(f"Erro ao processar o arquivo {caminho_pdf_original}: {e}")

def processar_lote(caminho_pasta_origem, caminho_pasta_destino):
    """
    Processa todos os arquivos PDF em uma pasta e salva os resultados em outra.
    """
    if not os.path.exists(caminho_pasta_destino):
        os.makedirs(caminho_pasta_destino)

    for nome_arquivo in os.listdir(caminho_pasta_origem):
        if nome_arquivo.endswith('.pdf'):
            caminho_original = os.path.join(caminho_pasta_origem, nome_arquivo)
            nome_anonimizado = f"ANONIMIZADO_{nome_arquivo}"
            caminho_final = os.path.join(caminho_pasta_destino, nome_anonimizado)

            print(f"\nProcessando arquivo: {nome_arquivo}")
            tarjar_cpfs(caminho_original, caminho_final)
            print("-" * 30)

# Exemplo de uso para processamento em lote
# Mude esses caminhos para as suas pastas
pasta_entrada = 'PDFs_originais'
pasta_saida = 'PDFs_anonimizados'

# Crie essas pastas e coloque os PDFs na pasta 'PDFs_originais'
processar_lote(pasta_entrada, pasta_saida)


Processando arquivo: Contrato - Camila Pereira da Silva Arauijo.pdf
CPF encontrado e tarjado na página 1 de PDFs_originais/Contrato - Camila Pereira da Silva Arauijo.pdf: 623.127.401-25,
CPF encontrado e tarjado na página 1 de PDFs_originais/Contrato - Camila Pereira da Silva Arauijo.pdf: 030.006.631-77,
CPF encontrado e tarjado na página 1 de PDFs_originais/Contrato - Camila Pereira da Silva Arauijo.pdf: 030.006.631-77
Novo arquivo salvo: PDFs_anonimizados/ANONIMIZADO_Contrato - Camila Pereira da Silva Arauijo.pdf
------------------------------

Processando arquivo: Contrato - Cleber Augusto Fernandes de Oliveira.pdf
CPF encontrado e tarjado na página 1 de PDFs_originais/Contrato - Cleber Augusto Fernandes de Oliveira.pdf: 623.127.401-25,
CPF encontrado e tarjado na página 1 de PDFs_originais/Contrato - Cleber Augusto Fernandes de Oliveira.pdf: 788.961.341-53,
CPF encontrado e tarjado na página 1 de PDFs_originais/Contrato - Cleber Augusto Fernandes de Oliveira.pdf: 788.961.341-53
No